# The Server Inspector

The MCP Python SDK ships a **browser-based inspector** to test a server's tools/resources/prompts in isolation — no need to wire up a client or Claude. Great for a fast build → test → debug loop.

> Guide lesson — the inspector is an external browser tool, so there's nothing to run *in* this notebook. Run against **`cli-project-complete/`** (fully working) or your **`cli-project/`** starter after you've added the tools.

## Starting the inspector

Activate your venv, then from the project folder run:

```powershell
# from repo root
.\.venv\Scripts\Activate.ps1
cd building-with-the-claude-api\07-mcp\cli-project-complete
mcp dev mcp_server.py
```

It prints a local URL (the proxy runs on port **6277**, the UI opens on **6274** in current versions) — open it in your browser to load the **MCP Inspector** dashboard.

> ⚠️ **Prerequisite the course doesn't mention: `mcp dev` needs Node.js / `npx`.** Under the hood it launches the JavaScript MCP Inspector via `npx @modelcontextprotocol/inspector`. If `node`/`npx` aren't installed, `mcp dev` fails. Options:
> - Install Node.js (bundles `npx`), then `mcp dev mcp_server.py` works.
> - **No Node?** You can still confirm the server registered its tools using the import-based check in **`04-defining-tools.ipynb`** (`await mcp.list_tools()`), and you'll be able to exercise the tools end-to-end through the CLI once the client is wired (next lesson).

## Interface note

The inspector is actively developed, so your UI may not match any screenshot — but the flow (connect, then browse **Resources / Prompts / Tools**) stays the same.

## Connecting and testing tools

1. Click **Connect** (left) to launch/attach to your server.
2. Use the top nav — **Tools**, plus **Resources** and **Prompts** (the complete server has all three).
3. In **Tools**, click **List Tools** → you should see `read_doc_contents` and `edit_document`.
4. Select a tool → fill parameters → **Run Tool** → see the result.

**Try a read → edit → read chain** (proves the tools mutate state):

| Step | Tool | Args | Expect |
|------|------|------|--------|
| 1 | `read_doc_contents` | `doc_id = deposition.md` | "This deposition covers the testimony of Angela Smith, P.E." |
| 2 | `edit_document` | `doc_id = deposition.md`, `old_str = Angela Smith`, `new_str = Angela Jones` | success |
| 3 | `read_doc_contents` | `doc_id = deposition.md` | "...testimony of Angela **Jones**, P.E." |

(The edit persists only for the life of that server process — docs are in memory.)

## Why it matters — the dev loop

```
edit mcp_server.py  ->  test the tool in the inspector  ->  verify  ->  debug in isolation
```

The inspector removes the need to stand up a client + Claude just to check a tool works — you test the server directly, which gets essential as servers grow (and it's how you'd sanity-check `list_docs` / `fetch_doc` resources and the `format` prompt in later lessons before touching the client).

Next: **Implementing a client** — fill in `mcp_client.py`'s `list_tools` / `call_tool` so the CLI chatbot can actually use these server tools.